<a href="https://colab.research.google.com/github/alxmzr/Colab/blob/main/Luna_Sun_Intraday.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Привет-привет. Есть такая идея, которую надо обсудить. Допустим, программа Astrology 7.70. Я открыл конструкцию, и оставил только Солнце, и Сындент, и Сындент, и середина неба, MC, не знаю, как правильно называется, и есть идея, просто я накручу график, начиная с Midnight, с подножия, и посмотрю, где есть квадратуры, где есть текстиль, где есть другие, там, позиции. Интересно создать скрипт, который будет мне подсказывать без прокручивания диагонального круга. Где-то подсказывает, что в какой-то момент, в этот день, в конкретный день, каждый день мы будем смотреть на это. Будет образовываться, допустим, оппозиция к Сынденту, или к Сынденту, или к середине неба. И просто подоставлю точки на графике, и мы точно будем знать, где возможен этот разворот. Спасибо за внимание!

In [1]:
!pip install -q ephem
!pip install -q pytz
!pip install -q python-telegram-bot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.4/745.4 kB 12.9 MB/s eta 0:00:00


In [2]:
import ephem
from datetime import datetime, timedelta
import pytz  # Библиотека для работы с часовыми поясами

# Настройки
OBSERVER_LAT = '40.7128'  # Широта Нью-Йорка
OBSERVER_LON = '-74.0060'  # Долгота
NY_TZ = pytz.timezone('America/New_York')  # Часовой пояс NY
ORB = 2  # Допуск в градусах
ASPECTS = {0: "Conjunction", 90: "Square", 180: "Opposition"}

def get_planet_pos(planet, observer):
    if planet == "Sun":
        return ephem.Sun(observer).ra / ephem.degree
    elif planet == "Moon":
        return ephem.Moon(observer).ra / ephem.degree
    elif planet == "Asc":
        observer.pressure = 0
        observer.horizon = '-0:34'
        return observer.radec_of(0, 0)[0] / ephem.degree % 360
    elif planet == "MC":
        return observer.sidereal_time() * 15 / ephem.degree % 360
    elif planet == "DS":
        return (get_planet_pos("Asc", observer) + 180) % 360
    elif planet == "IC":
        return (get_planet_pos("MC", observer) + 180) % 360
    return 0

def check_aspects(date, observer):
    planets = {
        "Sun": get_planet_pos("Sun", observer),
        "Moon": get_planet_pos("Moon", observer),
        "Asc": get_planet_pos("Asc", observer),
        "MC": get_planet_pos("MC", observer),
        "DS": get_planet_pos("DS", observer),
        "IC": get_planet_pos("DS", observer)
    }

    results = []
    pairs = [
        ("Sun", "Moon"), ("Sun", "Asc"), ("Sun", "MC"), ("Sun", "DS"), ("Sun", "IC"),
        ("Moon", "Asc"), ("Moon", "MC"), ("Moon", "DS"), ("Moon", "IC"),
    ]

    for p1, p2 in pairs:
        angle = abs(planets[p1] - planets[p2]) % 360
        for aspect_angle, name in ASPECTS.items():
            if abs(angle - aspect_angle) <= ORB:
                results.append(
                    f"{date.strftime('%H:%M')}: {name} {p1}-{p2} ({angle:.1f}°)"
                )
    return results

# Устанавливаем время начала торгов NYSE в правильном часовом поясе
ny_time = datetime.now(NY_TZ).replace(hour=0, minute=00, second=0)
start_date = NY_TZ.localize(ny_time) if ny_time.tzinfo is None else ny_time

# Инициализация наблюдателя
observer = ephem.Observer()
observer.lat, observer.lon = OBSERVER_LAT, OBSERVER_LON

print(f"Анализ аспектов ({start_date.strftime('%Y-%m-%d %H:%M')} NY Time):")
for minute in range(0, 1440, 5):  # 6.5 часов, шаг 5 минут
    current_date = start_date + timedelta(minutes=minute)
    observer.date = current_date
    aspects = check_aspects(current_date, observer)
    if aspects:
        print(f"\n{current_date.strftime('%H:%M')}:")
        print("\n".join(aspects))

Анализ аспектов (2026-05-06 00:00 NY Time):

00:35:
00:35: Conjunction Sun-MC (1.2°)

00:50:
00:50: Conjunction Sun-Asc (1.1°)
00:50: Opposition Sun-DS (178.9°)
00:50: Opposition Sun-IC (178.9°)

00:55:
00:55: Conjunction Sun-Asc (0.1°)
00:55: Opposition Sun-DS (180.1°)
00:55: Opposition Sun-IC (180.1°)

01:00:
01:00: Conjunction Sun-Asc (1.4°)
01:00: Opposition Sun-DS (181.4°)
01:00: Opposition Sun-IC (181.4°)

02:25:
02:25: Opposition Moon-MC (181.0°)

02:35:
02:35: Square Sun-MC (90.0°)

04:25:
04:25: Opposition Moon-Asc (181.5°)
04:25: Square Moon-MC (90.5°)
04:25: Conjunction Moon-DS (1.5°)
04:25: Conjunction Moon-IC (1.5°)

04:30:
04:30: Opposition Moon-Asc (180.2°)
04:30: Conjunction Moon-DS (0.2°)
04:30: Conjunction Moon-IC (0.2°)

04:35:
04:35: Opposition Sun-MC (181.1°)
04:35: Opposition Moon-Asc (179.0°)
04:35: Conjunction Moon-DS (1.0°)
04:35: Conjunction Moon-IC (1.0°)

06:10:
06:10: Opposition Sun-MC (178.3°)

06:25:
06:25: Conjunction Moon-MC (0.0°)

06:50:
06:50: Square

In [3]:
# !!!!!!!!!!!!!!!!!       WORK !!!!!!!!!!!!!!!!!!!!!!!!

In [4]:
import ephem
from datetime import datetime, timedelta
import pytz

# Настройки
OBSERVER_LAT = '40.7128'  # Широта Нью-Йорка
OBSERVER_LON = '-74.0060'  # Долгота
NY_TZ = pytz.timezone('America/New_York')
ORB = 2  # Допуск в градусах
ASPECTS = {0: "⚡", 90: "⚠", 180: "☠"}  # Символы для визуализации
STEP_MINUTES = 15  # Шаг анализа в минутах

def get_planet_pos(planet, observer):
    if planet == "Sun":
        return ephem.Sun(observer).ra / ephem.degree
    elif planet == "Moon":
        return ephem.Moon(observer).ra / ephem.degree
    elif planet == "Asc":
        observer.pressure = 0
        observer.horizon = '-0:34'
        return observer.radec_of(0, 0)[0] / ephem.degree % 360
    elif planet == "MC":
        return observer.sidereal_time() * 15 / ephem.degree % 360
    elif planet == "DS":
        return (get_planet_pos("Asc", observer) + 180) % 360
    elif planet == "IC":
        return (get_planet_pos("MC", observer) + 180) % 360
    return 0

def check_aspects(date, observer):
    planets = {
        "Sun": get_planet_pos("Sun", observer),
        "Moon": get_planet_pos("Moon", observer),
        "Asc": get_planet_pos("Asc", observer),
        "MC": get_planet_pos("MC", observer),
        "DS": get_planet_pos("DS", observer),
        "IC": get_planet_pos("IC", observer)
    }

    results = []
    pairs = [("Sun", "Moon"), ("Sun", "Asc"), ("Sun", "MC"), ("Sun", "IC") ,("Sun", "DS"),
             ("Moon", "Asc"), ("Moon", "MC"), ("Moon", "IC"), ("Moon", "DS")]

    for p1, p2 in pairs:
        angle = abs(planets[p1] - planets[p2]) % 360
        for aspect_angle, symbol in ASPECTS.items():
            if abs(angle - aspect_angle) <= ORB:
                results.append(f"{symbol} {p1}-{p2} ({angle:.1f}°)")
    return results

# Устанавливаем время начала торгов NYSE
ny_time = datetime.now(NY_TZ).replace(hour=0, minute=00, second=0)
start_date = NY_TZ.localize(ny_time) if ny_time.tzinfo is None else ny_time

observer = ephem.Observer()
observer.lat, observer.lon = OBSERVER_LAT, OBSERVER_LON

print(f"Анализ аспектов ({start_date.strftime('%Y-%m-%d')}):")
print("Время  | Аспекты")
print("-------|--------")

for minute in range(0, 1440, STEP_MINUTES):  # 6.5 часов с шагом 15 минут
    current_date = start_date + timedelta(minutes=minute)
    observer.date = current_date
    aspects = check_aspects(current_date, observer)
    if aspects:
        print(f"{current_date.strftime('%H:%M')} | {' | '.join(aspects)}")

Анализ аспектов (2026-05-06):
Время  | Аспекты
-------|--------
01:00 | ⚡ Sun-Asc (1.4°) | ☠ Sun-DS (181.4°)
04:30 | ☠ Moon-Asc (180.2°) | ⚡ Moon-DS (0.2°)
07:00 | ⚠ Sun-Asc (91.4°)
10:45 | ⚠ Moon-Asc (88.7°)
13:00 | ☠ Sun-Asc (181.6°) | ⚡ Sun-DS (1.6°)
13:45 | ⚠ Sun-MC (88.9°)
15:45 | ☠ Sun-MC (180.0°) | ⚡ Sun-IC (0.0°)
17:00 | ⚡ Moon-Asc (1.2°) | ☠ Moon-DS (178.8°)
17:45 | ⚠ Sun-IC (91.2°)
19:00 | ⚠ Sun-DS (91.6°)
23:15 | ⚠ Moon-DS (89.2°)


In [5]:
# !!!!!!!!!!!!!!!!!!!!!! WORK !!!!!!!!!!!!!!!!!!!!!!!!!!

In [6]:
import ephem
from datetime import datetime, timedelta
import pytz
from telegram import Bot
import asyncio

# Настройки
OBSERVER_LAT = '40.7128'  # Широта Нью-Йорка
OBSERVER_LON = '-74.0060'  # Долгота
NY_TZ = pytz.timezone('America/New_York')
ORB = 2  # Допуск в градусах
STEP_MINUTES = 15  # Шаг анализа в минутах

# Настройки Telegram
TELEGRAM_TOKEN = "7756894791:AAHkNhPjxzPs2NmBk_FjertCzClSywQBIIY"
CHAT_ID = "1563343518"  # ID чата или канала

# Символы для аспектов
ASPECTS = {
    0: {"symbol": "⚡", "name": "Соединение"},
    90: {"symbol": "⚠", "name": "Квадратура"},
    180: {"symbol": "☠", "name": "Оппозиция"}
}

async def send_to_telegram(message):
    bot = Bot(token=TELEGRAM_TOKEN)
    await bot.send_message(chat_id=CHAT_ID, text=message)

def get_planet_pos(planet, observer):
    if planet == "Sun":
        return ephem.Sun(observer).ra / ephem.degree
    elif planet == "Moon":
        return ephem.Moon(observer).ra / ephem.degree
    elif planet == "Asc":
        observer.pressure = 0
        observer.horizon = '-0:34'
        return observer.radec_of(0, 0)[0] / ephem.degree % 360
    elif planet == "MC":
        return observer.sidereal_time() * 15 / ephem.degree % 360
    elif planet == "DS":
        return (get_planet_pos("Asc", observer) + 180) % 360
    return 0

def check_aspects(date, observer):
    planets = {
        "Sun": get_planet_pos("Sun", observer),
        "Moon": get_planet_pos("Moon", observer),
        "Asc": get_planet_pos("Asc", observer),
        "MC": get_planet_pos("MC", observer),
        "DS": get_planet_pos("DS", observer)
    }

    results = []
    pairs = [("Sun", "Moon"), ("Sun", "Asc"), ("Sun", "MC"), ("Sun", "DS"),
             ("Moon", "Asc"), ("Moon", "MC"), ("Moon", "DS")]

    for p1, p2 in pairs:
        angle = abs(planets[p1] - planets[p2]) % 360
        for aspect_angle, data in ASPECTS.items():
            if abs(angle - aspect_angle) <= ORB:
                results.append(
                    f"{data['symbol']} {data['name']} {p1}-{p2} ({angle:.1f}°)"
                )
    return results

def main():
    observer = ephem.Observer()
    observer.lat, observer.lon = OBSERVER_LAT, OBSERVER_LON
    ny_time = datetime.now(NY_TZ).replace(hour=9, minute=30, second=0)
    start_date = NY_TZ.localize(ny_time) if ny_time.tzinfo is None else ny_time

    # Отправляем заголовок в Telegram
    header = f"📊 Астротрейдинг NYSE {start_date.strftime('%d.%m.%Y')}\n"
    header += "⏰ Время | Аспекты\n"
    header += "------------------"
    send_to_telegram(header)

    for minute in range(0, 390, STEP_MINUTES):
        current_date = start_date + timedelta(minutes=minute)
        observer.date = current_date
        aspects = check_aspects(current_date, observer)

        if aspects:
            message = f"🕒 {current_date.strftime('%H:%M')}:\n" + "\n".join(aspects)
            send_to_telegram(message)

if __name__ == "__main__":
    main()

/tmp/ipykernel_5018/3065857451.py:76: RuntimeWarning: coroutine 'send_to_telegram' was never awaited
  send_to_telegram(header)
/tmp/ipykernel_5018/3065857451.py:85: RuntimeWarning: coroutine 'send_to_telegram' was never awaited
  send_to_telegram(message)


In [7]:
import ephem
from datetime import datetime, timedelta

# Настройки
OBSERVER_LAT = '40.7128'  # Широта Нью-Йорка (для NYSE)
OBSERVER_LON = '-74.0060'  # Долгота
ORB = 2  # Допуск в градусах
ASPECTS = {0: "Conjunction", 90: "Square", 180: "Opposition"}

def get_planet_pos(planet, observer):
    if planet == "Sun":
        return ephem.Sun(observer).ra / ephem.degree
    elif planet == "Moon":
        return ephem.Moon(observer).ra / ephem.degree
    elif planet == "Asc":
        return ephem.Observer().radec_of(0, 0)[0] / ephem.degree  # Упрощенно
    elif planet == "MC":
        return observer.sidereal_time() / ephem.degree * 15  # MC ~ RAMC
    elif planet == "DS":
        return (ephem.Observer().radec_of(0, 0)[0] / ephem.degree + 180) % 360
    return 0

def check_aspects(date, observer):
    planets = {"Sun": 0, "Moon": 0, "Asc": 0, "MC": 0, "DS": 0}
    for planet in planets:
        planets[planet] = get_planet_pos(planet, observer)

    results = []
    pairs = [("Sun", "Moon"), ("Sun", "Asc"), ("Sun", "MC"), ("Sun", "DS"),
             ("Moon", "Asc"), ("Moon", "MC"), ("Moon", "DS")]

    for p1, p2 in pairs:
        angle = abs(planets[p1] - planets[p2]) % 360
        for aspect_angle, name in ASPECTS.items():
            if abs(angle - aspect_angle) <= ORB:
                results.append(
                    f"{date.strftime('%H:%M')}: {name} {p1}-{p2} ({angle:.1f}°)"
                )
    return results

# Запуск
observer = ephem.Observer()
observer.lat, observer.lon = OBSERVER_LAT, OBSERVER_LON
start_date = datetime.now().replace(hour=0, minute=00, second=0)  # Начало торгов

for minute in range(0, 1440, 5):  # 6.5 часов (NYSE), шаг 5 минут
    current_date = start_date + timedelta(minutes=minute)
    observer.date = current_date
    aspects = check_aspects(current_date, observer)
    if aspects:
        print(f"\n{current_date.strftime('%H:%M')}:")
        print("\n".join(aspects))


03:00:
03:00: Conjunction Sun-MC (1.7°)

04:25:
04:25: Square Moon-MC (88.6°)

06:25:
06:25: Opposition Moon-MC (179.0°)

06:35:
06:35: Square Sun-MC (90.0°)

08:35:
08:35: Opposition Sun-MC (181.2°)

10:10:
10:10: Opposition Sun-MC (178.3°)

10:25:
10:25: Conjunction Moon-MC (0.0°)

12:25:
12:25: Square Moon-MC (90.4°)

14:10:
14:10: Conjunction Sun-MC (0.6°)

14:25:
14:25: Opposition Moon-MC (179.3°)

16:10:
16:10: Square Sun-MC (91.8°)

17:45:
17:45: Square Sun-MC (88.9°)

18:25:
18:25: Conjunction Moon-MC (0.6°)

19:45:
19:45: Opposition Sun-MC (180.1°)

20:25:
20:25: Square Moon-MC (90.4°)

22:25:
22:25: Opposition Moon-MC (180.1°)


In [8]:
#++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import ephem
from datetime import datetime, timedelta

# Настройки
OBSERVER_LAT = '40.7128'  # Широта Нью-Йорка (для NYSE)
OBSERVER_LON = '-74.0060'  # Долгота
ORB = 2  # Допуск в градусах
ASPECTS = {0: "Conjunction", 90: "Square", 180: "Opposition"}

def get_planet_pos(planet, observer):
    if planet == "Sun":
        return ephem.Sun(observer).ra / ephem.degree
    elif planet == "Moon":
        return ephem.Moon(observer).ra / ephem.degree
    elif planet == "Asc":
        observer.pressure = 0  # Убираем рефракцию
        observer.horizon = '-0:34'  # Стандартный горизонт
        # Правильный расчёт асцендента
        asc_angle = observer.radec_of(0, 0)[0] / ephem.degree
        return asc_angle % 360
    elif planet == "MC":
        # Середина неба (MC) - это прямое восхождение меридиана
        mc_angle = observer.sidereal_time() * 15 / ephem.degree  # Переводим в градусы
        return mc_angle % 360
    elif planet == "DS":
        # Десцендент = Асцендент + 180°
        asc = get_planet_pos("Asc", observer)
        return (asc + 180) % 360
    return 0

def check_aspects(date, observer):
    planets = {
        "Sun": get_planet_pos("Sun", observer),
        "Moon": get_planet_pos("Moon", observer),
        "Asc": get_planet_pos("Asc", observer),
        "MC": get_planet_pos("MC", observer),
        "DS": get_planet_pos("DS", observer)
    }

    # Проверяем, что все позиции рассчитаны
    for planet, pos in planets.items():
        if pos is None:
            print(f"Warning: Не удалось рассчитать позицию для {planet}")
            return []

    results = []
    pairs = [
        ("Sun", "Moon"), ("Sun", "Asc"), ("Sun", "MC"), ("Sun", "DS"),
        ("Moon", "Asc"), ("Moon", "MC"), ("Moon", "DS")
    ]

    for p1, p2 in pairs:
        angle = abs(planets[p1] - planets[p2]) % 360
        for aspect_angle, name in ASPECTS.items():
            if abs(angle - aspect_angle) <= ORB:
                results.append(
                    f"{date.strftime('%H:%M')}: {name} {p1}-{p2} ({angle:.1f}°)"
                )
    return results

# Запуск
observer = ephem.Observer()
observer.lat, observer.lon = OBSERVER_LAT, OBSERVER_LON
start_date = datetime.now().replace(hour=0, minute=00, second=0)  # Начало торгов NYSE

print(f"Анализ аспектов для {start_date.date()} с 09:30 до 16:00 (NYSE)")
for minute in range(0, 1440, 5):  # 6.5 часов (NYSE), шаг 5 минут
    current_date = start_date + timedelta(minutes=minute)
    observer.date = current_date
    aspects = check_aspects(current_date, observer)
    if aspects:
        print(f"\n{current_date.strftime('%H:%M')}:")
        print("\n".join(aspects))

Анализ аспектов для 2026-05-06 с 09:30 до 16:00 (NYSE)

00:25:
00:25: Square Moon-MC (91.7°)

02:15:
02:15: Square Moon-DS (91.4°)

02:20:
02:20: Square Moon-DS (90.2°)

02:25:
02:25: Conjunction Moon-MC (1.7°)
02:25: Square Moon-DS (89.0°)

03:00:
03:00: Conjunction Sun-MC (1.6°)

04:35:
04:35: Conjunction Sun-MC (1.2°)

04:50:
04:50: Conjunction Sun-Asc (1.1°)
04:50: Opposition Sun-DS (178.9°)

04:55:
04:55: Conjunction Sun-Asc (0.1°)
04:55: Opposition Sun-DS (180.1°)

05:00:
05:00: Conjunction Sun-Asc (1.4°)
05:00: Opposition Sun-DS (181.4°)

06:25:
06:25: Opposition Moon-MC (181.1°)

06:35:
06:35: Square Sun-MC (90.0°)

08:25:
08:25: Opposition Moon-Asc (181.5°)
08:25: Square Moon-MC (90.6°)
08:25: Conjunction Moon-DS (1.5°)

08:30:
08:30: Opposition Moon-Asc (180.2°)
08:30: Conjunction Moon-DS (0.2°)

08:35:
08:35: Opposition Sun-MC (181.1°)
08:35: Opposition Moon-Asc (179.0°)
08:35: Conjunction Moon-DS (1.0°)

10:10:
10:10: Opposition Sun-MC (178.3°)

10:25:
10:25: Conjunction Mo

In [9]:
#any sate

In [10]:
import ephem
from datetime import datetime, timedelta

# Настройки
OBSERVER_LAT = '40.7128'  # Широта Нью-Йорка (для NYSE)
OBSERVER_LON = '-74.0060'  # Долгота
ORB = 2  # Допуск в градусах
ASPECTS = {0: "Conjunction", 90: "Square", 180: "Opposition"}

def get_planet_pos(planet, observer):
    if planet == "Sun":
        return ephem.Sun(observer).ra / ephem.degree
    elif planet == "Moon":
        return ephem.Moon(observer).ra / ephem.degree
    elif planet == "Asc":
        observer.pressure = 0  # Убираем рефракцию
        observer.horizon = '-0:34'  # Стандартный горизонт
        # Правильный расчёт асцендента
        asc_angle = observer.radec_of(0, 0)[0] / ephem.degree
        return asc_angle % 360
    elif planet == "MC":
        # Середина неба (MC) - это прямое восхождение меридиана
        mc_angle = observer.sidereal_time() * 15 / ephem.degree  # Переводим в градусы
        return mc_angle % 360
    elif planet == "DS":
        # Десцендент = Асцендент + 180°
        asc = get_planet_pos("Asc", observer)
        return (asc + 180) % 360
    return 0

def check_aspects(date, observer):
    planets = {
        "Sun": get_planet_pos("Sun", observer),
        "Moon": get_planet_pos("Moon", observer),
        "Asc": get_planet_pos("Asc", observer),
        "MC": get_planet_pos("MC", observer),
        "DS": get_planet_pos("DS", observer)
    }

    # Проверяем, что все позиции рассчитаны
    for planet, pos in planets.items():
        if pos is None:
            print(f"Warning: Не удалось рассчитать позицию для {planet}")
            return []

    results = []
    pairs = [
        ("Sun", "Moon"), ("Sun", "Asc"), ("Sun", "MC"), ("Sun", "DS"),
        ("Moon", "Asc"), ("Moon", "MC"), ("Moon", "DS")
    ]

    for p1, p2 in pairs:
        angle = abs(planets[p1] - planets[p2]) % 360
        for aspect_angle, name in ASPECTS.items():
            if abs(angle - aspect_angle) <= ORB:
                results.append(
                    f"{date.strftime('%H:%M')}: {name} {p1}-{p2} ({angle:.1f}°)"
                )
    return results

# Запуск
observer = ephem.Observer()
observer.lat, observer.lon = OBSERVER_LAT, OBSERVER_LON

start_date = datetime.strptime("17/07/25 0:00", "%d/%m/%y %H:%M")

print(f"Анализ аспектов для {start_date.date()}")
for minute in range(0, 1440, 5):  # 6.5 часов (NYSE), шаг 5 минут
    current_date = start_date + timedelta(minutes=minute)
    observer.date = current_date
    aspects = check_aspects(current_date, observer)
    if aspects:
        print(f"\n{current_date.strftime('%H:%M')}:")
        print("\n".join(aspects))

Анализ аспектов для 2025-07-17

00:30:
00:30: Opposition Moon-MC (178.5°)

00:35:
00:35: Square Sun-MC (91.0°)

02:10:
02:10: Square Sun-MC (88.1°)

04:00:
04:00: Square Moon-Asc (88.3°)

04:05:
04:05: Square Moon-Asc (89.5°)

04:10:
04:10: Opposition Sun-MC (179.3°)
04:10: Square Moon-Asc (90.7°)

04:15:
04:15: Square Moon-Asc (91.9°)

04:30:
04:30: Conjunction Moon-MC (1.4°)

05:00:
05:00: Conjunction Sun-Asc (1.0°)
05:00: Opposition Sun-DS (179.0°)

05:05:
05:05: Conjunction Sun-Asc (0.2°)
05:05: Opposition Sun-DS (180.2°)

05:10:
05:10: Conjunction Sun-Asc (1.5°)
05:10: Opposition Sun-DS (181.5°)

06:10:
06:10: Square Sun-MC (89.6°)

06:30:
06:30: Square Moon-MC (88.9°)

08:10:
08:10: Conjunction Sun-MC (1.6°)

08:30:
08:30: Opposition Moon-MC (179.4°)

09:45:
09:45: Conjunction Sun-MC (1.3°)

10:10:
10:10: Opposition Moon-Asc (178.7°)
10:10: Conjunction Moon-DS (1.3°)

10:15:
10:15: Opposition Moon-Asc (180.0°)
10:15: Conjunction Moon-DS (0.0°)

10:20:
10:20: Opposition Moon-Asc (

In [11]:
from datetime import datetime, timedelta

# Примерный список событий
astro_events = [
    {'datetime': datetime(2025, 7, 1, 3, 15), 'event': 'Square Moon-Asc (91.1°)'},
    {'datetime': datetime(2025, 7, 1, 3, 15), 'event': 'Square Moon-DS (88.9°)'},
    # и т.д.
]

def find_astro_events(user_input_time, window_minutes=15):
    dt = datetime.strptime(user_input_time, '%d/%m/%y %H:%M')
    start = dt - timedelta(minutes=window_minutes)
    end = dt + timedelta(minutes=window_minutes)

    nearby_events = [
        e['event'] for e in astro_events
        if start <= e['datetime'] <= end
    ]

    return nearby_events

# Пример
user_time = '17/07/25 03:45'
events = find_astro_events(user_time)
print(events)


[]


In [12]:
#Any date
import ephem
from datetime import datetime, timedelta
import pytz  # Библиотека для работы с часовыми поясами

# Настройки
OBSERVER_LAT = '40.7128'  # Широта Нью-Йорка
OBSERVER_LON = '-74.0060'  # Долгота
NY_TZ = pytz.timezone('America/New_York')  # Часовой пояс NY
ORB = 2  # Допуск в градусах
ASPECTS = {0: "Conjunction", 90: "Square", 180: "Opposition"}

def get_planet_pos(planet, observer):
    if planet == "Sun":
        return ephem.Sun(observer).ra / ephem.degree
    elif planet == "Moon":
        return ephem.Moon(observer).ra / ephem.degree
    elif planet == "Asc":
        observer.pressure = 0
        observer.horizon = '-0:34'
        return observer.radec_of(0, 0)[0] / ephem.degree % 360
    elif planet == "MC":
        return observer.sidereal_time() * 15 / ephem.degree % 360
    elif planet == "DS":
        return (get_planet_pos("Asc", observer) + 180) % 360
    elif planet == "IC":
        return (get_planet_pos("IC", observer) + 180) % 360
    return 0

def check_aspects(date, observer):
    planets = {
        "Sun": get_planet_pos("Sun", observer),
        "Moon": get_planet_pos("Moon", observer),
        "Asc": get_planet_pos("Asc", observer),
        "MC": get_planet_pos("MC", observer),
        "DS": get_planet_pos("DS", observer),
        "IC": get_planet_pos("DS", observer)
    }

    results = []
    pairs = [
        ("Sun", "Moon"), ("Sun", "Asc"), ("Sun", "MC"), ("Sun", "DS"), ("Sun", "IC"),
        ("Moon", "Asc"), ("Moon", "MC"), ("Moon", "DS"), ("Moon", "IC")]

    for p1, p2 in pairs:
        angle = abs(planets[p1] - planets[p2]) % 360
        for aspect_angle, name in ASPECTS.items():
            if abs(angle - aspect_angle) <= ORB:
                results.append(
                    f"{date.strftime('%H:%M')}: {name} {p1}-{p2} ({angle:.1f}°)"
                )
    return results

# Устанавливаем свою дату (год, месяц, день)
custom_date = datetime(2025, 7, 17)  # ← Здесь укажите нужную дату
ny_time = NY_TZ.localize(custom_date.replace(hour=0, minute=0, second=0))

# Инициализация наблюдателя
observer = ephem.Observer()
observer.lat, observer.lon = OBSERVER_LAT, OBSERVER_LON

print(f"Анализ аспектов для NYSE ({ny_time.strftime('%Y-%m-%d %H:%M')} NY Time):")
for minute in range(0, 1440, 5):  # 6.5 часов, шаг 5 минут
    current_date = ny_time + timedelta(minutes=minute)
    observer.date = current_date
    aspects = check_aspects(current_date, observer)
    if aspects:
        print(f"\n{current_date.strftime('%H:%M')}:")
        print("\n".join(aspects))

Анализ аспектов для NYSE (2025-07-17 00:00 NY Time):

00:00:
00:00: Square Moon-Asc (88.3°)

00:05:
00:05: Square Moon-Asc (89.5°)

00:10:
00:10: Opposition Sun-MC (179.3°)
00:10: Square Moon-Asc (90.7°)

00:15:
00:15: Square Moon-Asc (91.9°)

00:30:
00:30: Conjunction Moon-MC (1.4°)

01:00:
01:00: Conjunction Sun-Asc (1.0°)
01:00: Opposition Sun-DS (179.0°)
01:00: Opposition Sun-IC (179.0°)

01:05:
01:05: Conjunction Sun-Asc (0.2°)
01:05: Opposition Sun-DS (180.2°)
01:05: Opposition Sun-IC (180.2°)

01:10:
01:10: Conjunction Sun-Asc (1.5°)
01:10: Opposition Sun-DS (181.5°)
01:10: Opposition Sun-IC (181.5°)

02:10:
02:10: Square Sun-MC (89.6°)

02:30:
02:30: Square Moon-MC (88.9°)

04:10:
04:10: Conjunction Sun-MC (1.6°)

04:30:
04:30: Opposition Moon-MC (179.4°)

05:45:
05:45: Conjunction Sun-MC (1.3°)

06:10:
06:10: Opposition Moon-Asc (178.7°)
06:10: Conjunction Moon-DS (1.3°)
06:10: Conjunction Moon-IC (1.3°)

06:15:
06:15: Opposition Moon-Asc (180.0°)
06:15: Conjunction Moon-DS (0

In [49]:
import ephem
from datetime import datetime, timedelta
import pytz
import pandas as pd
import math

# Настройки
NY_TZ = pytz.timezone('America/New_York')
OBSERVER_LAT = '40.7128'
OBSERVER_LON = '-74.0060'
SCAN_ORB = 0.5  # Ищем точное касание (0.5 градуса)

# Список наиболее эффективных планет из бэктеста
TARGET_PLANETS = ["Moon", "Jupiter", "Sun"]
TARGET_ANGLES = ["Asc", "MC"]
TARGET_ASPECTS = {0: "Conj", 90: "Square", 180: "Opp"}

def get_angles_precise(obs):
    ra_mc = obs.sidereal_time()
    eps = 23.439 * math.pi / 180
    mc_lon = math.atan2(math.sin(ra_mc), math.cos(ra_mc) * math.cos(eps))
    lat = float(obs.lat) * math.pi / 180
    # Уточненный расчет Асцендента
    asc_lon = math.atan2(math.cos(ra_mc), -(math.sin(ra_mc) * math.cos(eps) + math.tan(lat) * math.sin(eps)))
    return (mc_lon * 180 / math.pi) % 360, (asc_lon * 180 / math.pi) % 360

def scan_next_24h():
    obs = ephem.Observer()
    obs.lat, obs.lon = OBSERVER_LAT, OBSERVER_LON

    # Текущее время в NY
    now_ny = datetime.now(NY_TZ)
    events = []

    print(f"🕒 Сканирование разворотных точек на период: {now_ny.strftime('%Y-%m-%d %H:%M')} --- {(now_ny + timedelta(hours=24)).strftime('%H:%M')} (NY Time)")

    # Шаг сканирования - 1 минута для высокой точности
    for m in range(0, 1440):
        check_time = now_ny + timedelta(minutes=m)
        obs.date = ephem.Date(check_time.astimezone(pytz.utc))

        mc_lon, asc_lon = get_angles_precise(obs)
        angles = {"Asc": asc_lon, "MC": mc_lon}

        planets_lon = {
            "Sun": ephem.Ecliptic(ephem.Sun(obs)).lon / ephem.degree,
            "Moon": ephem.Ecliptic(ephem.Moon(obs)).lon / ephem.degree,
            "Jupiter": ephem.Ecliptic(ephem.Jupiter(obs)).lon / ephem.degree
        }

        for p_name, p_lon in planets_lon.items():
            for a_name, a_lon in angles.items():
                diff = abs(p_lon - a_lon) % 360
                if diff > 180: diff = 360 - diff

                for asp_val, asp_name in TARGET_ASPECTS.items():
                    if abs(diff - asp_val) < 0.1: # Порог срабатывания 0.1 градуса (пик)
                        # Проверка, чтобы не дублировать одно и то же событие каждую минуту
                        event_desc = f"{asp_name} {p_name} to {a_name}"
                        if not events or events[-1]['Event'] != event_desc or (check_time - events[-1]['_raw_time']).seconds > 600:
                            events.append({
                                'Time (NY)': check_time.strftime('%H:%M'),
                                'Event': event_desc,
                                '_raw_time': check_time
                            })

    if events:
        df_events = pd.DataFrame(events).drop(columns=['_raw_time'])
        display(df_events)
    else:
        print("❌ Точных разворотных точек по выбранным параметрам не найдено.")

scan_next_24h()

🕒 Сканирование разворотных точек на период: 2026-05-06 05:32 --- 05:32 (NY Time)


,Time (NY),Event
0,06:50,Conj Sun to Asc
1,10:32,Square Moon to MC
2,10:37,Opp Moon to Asc
3,11:09,Square Jupiter to MC
4,11:20,Conj Jupiter to Asc
5,12:51,Conj Sun to MC
6,13:10,Square Sun to Asc
7,16:42,Square Moon to Asc
8,16:51,Opp Moon to MC
9,17:09,Square Jupiter to Asc


In [53]:
import ephem
from datetime import datetime, timedelta
import pytz
import pandas as pd
import math
from IPython.display import display, HTML

# --- CONFIGURATION ---
NY_TZ = pytz.timezone('America/New_York')
OBSERVER_LAT = '40.7128'
OBSERVER_LON = '-74.0060'

# Most Profitable Patterns from our Research
# High Win Rate (>65%) or High Avg Return
PROFITABLE_PATTERNS = {
    'Conj Moon-MC': '🔥 High Win Rate (Forex 87-93%) - Potential Reversal/Trend Peak',
    'Conj Jupiter-Asc': '🚀 Strong Bullish (SPY 75%, BTC 60%) - Growth Signal',
    'Opp Moon-Asc': '⚡ Reversal Signal (Forex/Stocks) - Volatility Spike',
    'Square Jupiter-MC': '💎 Institutional Move (SPY 70%) - Mid-day Pivot',
    'Square Moon-Asc': '⚠️ Scalp Opportunity - Minor Reversal'
}

def get_angles_precise(obs):
    ra_mc = obs.sidereal_time()
    eps = 23.439 * math.pi / 180
    mc_lon = math.atan2(math.sin(ra_mc), math.cos(ra_mc) * math.cos(eps))
    lat = float(obs.lat) * math.pi / 180
    asc_lon = math.atan2(math.cos(ra_mc), -(math.sin(ra_mc) * math.cos(eps) + math.tan(lat) * math.sin(eps)))
    return (mc_lon * 180 / math.pi) % 360, (asc_lon * 180 / math.pi) % 360

def scan_signals(days=1):
    obs = ephem.Observer()
    obs.lat, obs.lon = OBSERVER_LAT, OBSERVER_LON
    now_ny = datetime.now(NY_TZ)
    events = []

    # Scan range: per minute for 24h, or per 5 mins for weekly to save memory
    step = 1 if days == 1 else 10
    total_steps = (1440 * days) // step

    for i in range(total_steps):
        check_time = now_ny + timedelta(minutes=i * step)
        obs.date = ephem.Date(check_time.astimezone(pytz.utc))

        mc_lon, asc_lon = get_angles_precise(obs)
        angles = {"Asc": asc_lon, "MC": mc_lon}
        planets = {
            "Sun": ephem.Ecliptic(ephem.Sun(obs)).lon / ephem.degree,
            "Moon": ephem.Ecliptic(ephem.Moon(obs)).lon / ephem.degree,
            "Jupiter": ephem.Ecliptic(ephem.Jupiter(obs)).lon / ephem.degree
        }

        for p_name, p_lon in planets.items():
            for a_name, a_lon in angles.items():
                diff = abs(p_lon - a_lon) % 360
                if diff > 180: diff = 360 - diff

                for asp_val, asp_name in {0: "Conj", 90: "Square", 180: "Opp"}.items():
                    if abs(diff - asp_val) < 0.15:
                        pattern = f"{asp_name} {p_name}-{a_name}"
                        if pattern in PROFITABLE_PATTERNS:
                            if not events or events[-1]['Pattern'] != pattern or (check_time - events[-1]['_dt']).seconds > 3600:
                                events.append({
                                    'Date': check_time.strftime('%Y-%m-%d'),
                                    'Time (NY)': check_time.strftime('%H:%M'),
                                    'Pattern': pattern,
                                    'Insight': PROFITABLE_PATTERNS[pattern],
                                    '_dt': check_time
                                })
    return pd.DataFrame(events).drop(columns=['_dt']) if events else pd.DataFrame()

# --- DASHBOARD GENERATION ---
print("Generating Astro-Trading Dashboard...")
df_intraday = scan_signals(1)
df_weekly = scan_signals(7)

display(HTML("<h2>🔭 Astro-Signal Dashboard</h2>"))

if not df_intraday.empty:
    display(HTML("<h3>📌 Intraday Signals (Next 24 Hours)</h3>"))
    display(df_intraday)
else:
    print("No high-probability intraday signals found for the next 24h.")

if not df_weekly.empty:
    display(HTML("<h3>📅 Weekly Outlook (Next 7 Days)</h3>"))
    # Group by date for cleaner view
    display(df_weekly)
else:
    print("No weekly signals found.")

display(HTML("<p style='color: gray;'><i>Note: Times are in NY Time. Signals are based on historical Win Rate > 65% observed in 2023-2024 backtests.</i></p>"))

Generating Astro-Trading Dashboard...


,Date,Time (NY),Pattern,Insight
0,2026-05-06,10:37,Opp Moon-Asc,⚡ Reversal Signal (Forex/Stocks) - Volatility ...
1,2026-05-06,11:09,Square Jupiter-MC,💎 Institutional Move (SPY 70%) - Mid-day Pivot
2,2026-05-06,11:20,Conj Jupiter-Asc,"🚀 Strong Bullish (SPY 75%, BTC 60%) - Growth S..."
3,2026-05-06,16:42,Square Moon-Asc,⚠️ Scalp Opportunity - Minor Reversal
4,2026-05-06,23:07,Square Jupiter-MC,💎 Institutional Move (SPY 70%) - Mid-day Pivot
5,2026-05-07,05:02,Square Moon-Asc,⚠️ Scalp Opportunity - Minor Reversal
6,2026-05-07,05:15,Conj Moon-MC,🔥 High Win Rate (Forex 87-93%) - Potential Rev...


,Date,Time (NY),Pattern,Insight
0,2026-05-07,17:24,Square Moon-Asc,⚠️ Scalp Opportunity - Minor Reversal
1,2026-05-07,23:04,Square Jupiter-MC,💎 Institutional Move (SPY 70%) - Mid-day Pivot
2,2026-05-08,05:44,Square Moon-Asc,⚠️ Scalp Opportunity - Minor Reversal
3,2026-05-08,11:14,Conj Jupiter-Asc,"🚀 Strong Bullish (SPY 75%, BTC 60%) - Growth S..."
4,2026-05-08,12:14,Opp Moon-Asc,⚡ Reversal Signal (Forex/Stocks) - Volatility ...
5,2026-05-10,22:54,Square Jupiter-MC,💎 Institutional Move (SPY 70%) - Mid-day Pivot
6,2026-05-11,11:04,Conj Jupiter-Asc,"🚀 Strong Bullish (SPY 75%, BTC 60%) - Growth S..."
7,2026-05-11,14:34,Opp Moon-Asc,⚡ Reversal Signal (Forex/Stocks) - Volatility ...
8,2026-05-12,09:04,Square Moon-Asc,⚠️ Scalp Opportunity - Minor Reversal


### 📅 30-Day Astro-Trading Forecast
This scanner extends our probability model to the next 30 days, filtering for patterns that showed >65% Win Rates in our 2023-2024 study.

In [54]:
# Run scan for 30 days
df_monthly = scan_signals(days=30)

if not df_monthly.empty:
    display(HTML("<h3>🗓️ 30-Day High-Probability Outlook</h3>"))
    # Display grouped by date for better readability
    display(df_monthly)
else:
    print("No high-probability signals found in the next 30 days.")

,Date,Time (NY),Pattern,Insight
0,2026-05-06,10:37,Opp Moon-Asc,⚡ Reversal Signal (Forex/Stocks) - Volatility ...
1,2026-05-06,23:07,Square Jupiter-MC,💎 Institutional Move (SPY 70%) - Mid-day Pivot
2,2026-05-07,11:17,Conj Jupiter-Asc,"🚀 Strong Bullish (SPY 75%, BTC 60%) - Growth S..."
3,2026-05-09,22:57,Square Jupiter-MC,💎 Institutional Move (SPY 70%) - Mid-day Pivot
4,2026-05-10,07:17,Square Moon-Asc,⚠️ Scalp Opportunity - Minor Reversal
5,2026-05-10,07:37,Conj Moon-MC,🔥 High Win Rate (Forex 87-93%) - Potential Rev...
6,2026-05-10,11:07,Conj Jupiter-Asc,"🚀 Strong Bullish (SPY 75%, BTC 60%) - Growth S..."
7,2026-05-12,22:47,Square Jupiter-MC,💎 Institutional Move (SPY 70%) - Mid-day Pivot
8,2026-05-13,10:57,Conj Jupiter-Asc,"🚀 Strong Bullish (SPY 75%, BTC 60%) - Growth S..."
9,2026-05-15,22:37,Square Jupiter-MC,💎 Institutional Move (SPY 70%) - Mid-day Pivot


### 📊 Исследование Форекс: Астро-сигналы vs Дневные Экстремумы (HOD/LOD)
Этот анализ выявляет связь между планетарными аспектами к углам (Asc/MC) и моментами достижения абсолютного максимума (High) и минимума (Low) торгового дня для основных валютных пар.

### 📊 Исследование Индексов: Астро-сигналы vs Экстремумы (SPY, QQQ, DIA)
Анализ корреляции между планетарными аспектами и временем формирования максимумов (HOD) и минимумов (LOD) торговой сессии на американском фондовом рынке.

In [59]:
import yfinance as yf
import pandas as pd
import ephem
from datetime import datetime, timedelta
import pytz
import math
from IPython.display import display

# Настройки
INDEX_ETFS = {'SPY': 'S&P 500', 'QQQ': 'NASDAQ 100', 'DIA': 'Dow Jones'}
NY_TZ = pytz.timezone('America/New_York')
OBS_LAT, OBS_LON = '40.7128', '-74.0060'
TEST_ORB = 3.5 # Допуск в градусах для поиска корреляции

def get_angles_precise(obs):
    ra_mc = obs.sidereal_time()
    eps = 23.439 * math.pi / 180
    mc_lon = math.atan2(math.sin(ra_mc), math.cos(ra_mc) * math.cos(eps))
    lat = float(obs.lat) * math.pi / 180
    asc_lon = math.atan2(math.cos(ra_mc), -(math.sin(ra_mc) * math.cos(eps) + math.tan(lat) * math.sin(eps)))
    return (mc_lon * 180 / math.pi) % 360, (asc_lon * 180 / math.pi) % 360

def run_extreme_research(days=150):
    all_hits = []
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)

    for ticker, name in INDEX_ETFS.items():
        print(f"🔍 Анализируем {name} ({ticker})...")
        # Используем 1-часовые данные
        data = yf.download(ticker, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='1h', progress=False, auto_adjust=True)

        if data.empty: continue

        # Группируем по дням
        for day, day_data in data.groupby(data.index.date):
            if len(day_data) < 5: continue

            # Находим время HOD и LOD
            hod_idx = day_data['High'].idxmax()
            lod_idx = day_data['Low'].idxmin()

            # Обработка случаев, если вернулось несколько одинаковых экстремумов
            if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
            if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]

            extremes = [('High (HOD)', hod_idx), ('Low (LOD)', lod_idx)]

            for ext_label, ext_time in extremes:
                obs = ephem.Observer()
                obs.lat, obs.lon = OBS_LAT, OBS_LON
                # Конвертация в UTC для ephem
                obs.date = ephem.Date(ext_time.to_pydatetime().astimezone(pytz.utc))

                mc, asc = get_angles_precise(obs)
                angles = {'Asc': asc, 'MC': mc}

                planets = {
                    "Sun": ephem.Ecliptic(ephem.Sun(obs)).lon / ephem.degree,
                    "Moon": ephem.Ecliptic(ephem.Moon(obs)).lon / ephem.degree,
                    "Jupiter": ephem.Ecliptic(ephem.Jupiter(obs)).lon / ephem.degree
                }

                for p_name, p_lon in planets.items():
                    for a_name, a_lon in angles.items():
                        diff = abs(p_lon - a_lon) % 360
                        if diff > 180: diff = 360 - diff

                        for asp_val, asp_name in {0: 'Conj', 90: 'Square', 180: 'Opp'}.items():
                            if abs(diff - asp_val) <= TEST_ORB:
                                all_hits.append({
                                    'Index': ticker,
                                    'Extreme': ext_label,
                                    'Aspect': f"{asp_name} {p_name}-{a_name}",
                                    'Error': round(abs(diff - asp_val), 2)
                                })

    df_res = pd.DataFrame(all_hits)
    if df_res.empty: return None

    summary = df_res.groupby(['Index', 'Extreme', 'Aspect']).size().reset_index(name='Frequency')
    return summary.sort_values(by=['Index', 'Frequency'], ascending=[True, False])

# Выполнение
research_results = run_extreme_research(150)

if research_results is not None:
    print("\n✅ ИССЛЕДОВАНИЕ ЗАВЕРШЕНО")
    print("--- ТОП АСТРО-КОРРЕЛЯЦИЙ ДЛЯ ГЛАВНЫХ ТОЧЕК ДНЯ ---")
    display(research_results.head(30))
else:
    print("Недостаточно данных для формирования отчета.")

🔍 Анализируем S&P 500 (SPY)...


/tmp/ipykernel_5018/2234033651.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
/tmp/ipykernel_5018/2234033651.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]


🔍 Анализируем NASDAQ 100 (QQQ)...


/tmp/ipykernel_5018/2234033651.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
/tmp/ipykernel_5018/2234033651.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]


🔍 Анализируем Dow Jones (DIA)...


/tmp/ipykernel_5018/2234033651.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
/tmp/ipykernel_5018/2234033651.py:45: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]



✅ ИССЛЕДОВАНИЕ ЗАВЕРШЕНО
--- ТОП АСТРО-КОРРЕЛЯЦИЙ ДЛЯ ГЛАВНЫХ ТОЧЕК ДНЯ ---


,Index,Extreme,Aspect,Frequency
13,DIA,Low (LOD),Square Jupiter-Asc,5
15,DIA,Low (LOD),Square Moon-Asc,5
2,DIA,High (HOD),Opp Jupiter-MC,4
3,DIA,High (HOD),Opp Moon-Asc,4
7,DIA,High (HOD),Square Moon-MC,4
5,DIA,High (HOD),Square Jupiter-Asc,3
10,DIA,Low (LOD),Conj Moon-MC,3
11,DIA,Low (LOD),Opp Jupiter-MC,3
0,DIA,High (HOD),Conj Jupiter-Asc,2
1,DIA,High (HOD),Conj Moon-MC,2


### 📊 Исследование Сырьевых Рынков: Астро-сигналы vs Экстремумы (Золото, Серебро, Нефть)
Анализ корреляции аспектов для активов GLD (Gold), SLV (Silver) и USO (Oil).

### 📊 Сводное сравнение Win Rate: Крипто vs FX vs Индексы
Этот отчет объединяет результаты бэктестов для выявления наиболее устойчивых астрологических паттернов.

### 📅 Еженедельный Прогноз Высоковероятных Сигналов (7 Дней)
Этот отчет фильтрует только те паттерны, которые показали Win Rate > 65% в нашем анализе.

In [65]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
from IPython.display import display, HTML

# Настройки
NY_TZ = pytz.timezone('America/New_York')

# База данных сигналов на основе нашего бэктеста
SIGNAL_DATABASE = {
    'Conj Moon-MC': {'Asset': 'EURUSD, GBPUSD', 'WinRate': 90.6, 'Impact': 'High', 'Strategy': 'Trend exhaustion/Reversal'},
    'Square Jupiter-Asc': {'Asset': 'SPY, BTC-USD', 'WinRate': 66.2, 'Impact': 'Medium', 'Strategy': 'Bullish Institutional Push'},
    'Opp Moon-Asc': {'Asset': 'SPY, FX Pairs', 'WinRate': 64.5, 'Impact': 'Medium', 'Strategy': 'Volatility Spike'},
    'Square Moon-Asc': {'Asset': 'SPY', 'WinRate': 63.6, 'Impact': 'Low', 'Strategy': 'Scalp Reversal'},
    'Conj Jupiter-Asc': {'Asset': 'BTC-USD', 'WinRate': 61.0, 'Impact': 'Medium', 'Strategy': 'Growth Momentum'}
}

def generate_intraday_table():
    # Получаем сырые данные аспектов на ближайшие 24 часа
    # Используем существующую функцию scan_signals (предполагается, что она определена выше)
    df_raw = scan_signals(days=1)

    if df_raw.empty:
        return "На ближайшие 24 часа высоковероятных сигналов не найдено."

    report_list = []
    for _, row in df_raw.iterrows():
        pattern = row['Pattern']
        if pattern in SIGNAL_DATABASE:
            data = SIGNAL_DATABASE[pattern]
            report_list.append({
                'Time (NY)': row['Time (NY)'],
                'Signal': pattern,
                'Best Asset': data['Asset'],
                'Win Rate (%)': data['WinRate'],
                'Strategy': data['Strategy'],
                'Confidence': data['Impact']
            })

    if not report_list:
        return "Сигналы найдены, но их Win Rate ниже порогового значения (60%)."

    return pd.DataFrame(report_list)

# Генерация отчета
print(f"📊 ГЕНЕРАЦИЯ ИНТРАДЕЙ ОТЧЕТА: {datetime.now(NY_TZ).strftime('%Y-%m-%d')}")
result_df = generate_intraday_table()

if isinstance(result_df, pd.DataFrame):
    display(HTML("<h3>📈 Интрадей сигналы с высокой вероятностью (>60% WR)</h3>"))
    display(result_df.sort_values('Time (NY)'))
else:
    print(result_df)

📊 ГЕНЕРАЦИЯ ИНТРАДЕЙ ОТЧЕТА: 2026-05-06


,Time (NY),Signal,Best Asset,Win Rate (%),Strategy,Confidence
3,05:02,Square Moon-Asc,SPY,63.6,Scalp Reversal,Low
4,05:15,Conj Moon-MC,"EURUSD, GBPUSD",90.6,Trend exhaustion/Reversal,High
0,10:37,Opp Moon-Asc,"SPY, FX Pairs",64.5,Volatility Spike,Medium
1,11:20,Conj Jupiter-Asc,BTC-USD,61.0,Growth Momentum,Medium
2,16:42,Square Moon-Asc,SPY,63.6,Scalp Reversal,Low


In [64]:
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import display, HTML

# Словарь лучших стратегий на основе бэктеста (Win Rate > 60%)
SIGNAL_STATS = {
    'Conj Moon-MC': {'Market': 'FX (EURUSD, GBPUSD)', 'WinRate': '90.6%', 'Note': 'Разворот сессии'},
    'Square Jupiter-Asc': {'Market': 'Indices (SPY) / Crypto', 'WinRate': '66.2%', 'Note': 'Институциональный лонг'},
    'Opp Moon-Asc': {'Market': 'FX / Stocks', 'WinRate': '64.5%', 'Note': 'Всплеск волатильности'},
    'Square Moon-Asc': {'Market': 'Indices (SPY)', 'WinRate': '63.6%', 'Note': 'Скальп-разворот'},
    'Conj Jupiter-Asc': {'Market': 'Crypto (BTC)', 'WinRate': '61.0%', 'Note': 'Среднесрочный рост'}
}

def generate_pro_report(days=7):
    # Используем ранее определенную функцию scan_signals для поиска аспектных касаний
    raw_signals = scan_signals(days=days)

    if raw_signals.empty:
        return "Значимых сигналов с высоким Win Rate не найдено."

    # Обогащаем таблицу данными из статистики
    report_data = []
    for _, row in raw_signals.iterrows():
        pattern = row['Pattern']
        if pattern in SIGNAL_STATS:
            report_data.append({
                'Дата': row['Date'],
                'Время (NY)': row['Time (NY)'],
                'Сигнал': pattern,
                'Лучший Актив': SIGNAL_STATS[pattern]['Market'],
                'Win Rate': SIGNAL_STATS[pattern]['WinRate'],
                'Контекст': SIGNAL_STATS[pattern]['Note']
            })

    return pd.DataFrame(report_data)

# Генерация и вывод
print("📊 ФОРМИРОВАНИЕ ПРОФЕССИОНАЛЬНОГО АСТРО-ПРОГНОЗА...")
df_report = generate_pro_report(7)

if isinstance(df_report, pd.DataFrame):
    display(HTML("<h3>💎 Топ-сигналы на неделю (Win Rate > 60%)</h3>"))
    display(df_report.sort_values(by=['Дата', 'Время (NY)']))
else:
    print(df_report)

📊 ФОРМИРОВАНИЕ ПРОФЕССИОНАЛЬНОГО АСТРО-ПРОГНОЗА...


,Дата,Время (NY),Сигнал,Лучший Актив,Win Rate,Контекст
0,2026-05-07,05:03,Square Moon-Asc,Indices (SPY),63.6%,Скальп-разворот
1,2026-05-08,06:03,Conj Moon-MC,"FX (EURUSD, GBPUSD)",90.6%,Разворот сессии
2,2026-05-11,08:23,Conj Moon-MC,"FX (EURUSD, GBPUSD)",90.6%,Разворот сессии


In [63]:
import pandas as pd
from datetime import datetime, timedelta

# Фильтруем только топ-сигналы из нашего исследования
TOP_SIGNALS = {
    'Conj Moon-MC': '🚀 [FX] High Probability Reversal (90% WR)',
    'Square Jupiter-Asc': '💎 [Index/Crypto] Bullish Institutional Move (66% WR)',
    'Opp Moon-Asc': '⚠️ [Volatility] Potential Fast Spike/Flush'
}

def generate_weekly_report():
    # Используем существующую функцию сканирования для получения всех аспектов
    df_raw = scan_signals(days=7)

    if df_raw.empty:
        print("Значимых сигналов на ближайшую неделю не обнаружено.")
        return

    # Фильтруем по списку прибыльных паттернов
    df_top = df_raw[df_raw['Pattern'].isin(TOP_SIGNALS.keys())].copy()

    if df_top.empty:
        print("Сигналы с высоким Win Rate (>65%) на этой неделе отсутствуют.")
    else:
        # Добавляем наше описание к каждому сигналу
        df_top['Strategy'] = df_top['Pattern'].map(TOP_SIGNALS)

        # Оформление вывода
        print(f"--- ПРОГНОЗ С {datetime.now(NY_TZ).strftime('%d.%m')} ПО {(datetime.now(NY_TZ)+timedelta(days=7)).strftime('%d.%m')} ---")
        display(df_top[['Date', 'Time (NY)', 'Pattern', 'Strategy']])

generate_weekly_report()

Сигналы с высоким Win Rate (>65%) на этой неделе отсутствуют.


In [62]:
import pandas as pd

# Собираем данные из предыдущих расчетов (report для крипто/акций и summary для FX)
# Для чистоты эксперимента берем результаты из final_df (где есть Is_Win)

if 'final_df' in globals() and not final_df.empty:
    # Классифицируем активы по группам
    def classify_asset(asset):
        if 'USD' in asset: return 'Crypto' if '-' in asset else 'FX'
        return 'Index'

    comparison_df = final_df.copy()
    comparison_df['Market_Type'] = comparison_df['Asset'].apply(classify_asset)

    # Агрегация по типу рынка и аспекту
    cross_market_report = comparison_df.groupby(['Market_Type', 'Aspect']).agg(
        Samples=('Return_%', 'count'),
        Avg_Return=('Return_%', 'mean'),
        Win_Rate=('Is_Win', 'mean')
    ).reset_index()

    cross_market_report['Win_Rate_%'] = cross_market_report['Win_Rate'] * 100

    # Фильтруем значимые выборки (>15 событий)
    top_comparison = cross_market_report[cross_market_report['Samples'] > 15].sort_values(by='Win_Rate_%', ascending=False)

    print("--- СРАВНЕНИЕ ЭФФЕКТИВНОСТИ ПО РЫНКАМ ---")
    display(top_comparison.head(30))
else:
    print("Данные для сравнения не найдены. Пожалуйста, запустите ячейки с глобальным бэктестом.")

--- СРАВНЕНИЕ ЭФФЕКТИВНОСТИ ПО РЫНКАМ ---


,Market_Type,Aspect,Samples,Avg_Return,Win_Rate,Win_Rate_%
15,FX,Conj Moon-MC,32,0.328571,0.906250,90.625000
32,Index,Square Jupiter-Asc,24,0.241489,0.666667,66.666667
8,Crypto,Square Jupiter-Asc,35,0.923080,0.657143,65.714286
22,FX,Square Moon-Asc,66,0.087384,0.636364,63.636364
34,Index,Square Moon-Asc,33,0.155670,0.636364,63.636364
27,Index,Conj Moon-MC,16,0.250276,0.625000,62.500000
31,Index,Opp Moon-MC,20,0.077009,0.600000,60.000000
11,Crypto,Square Moon-MC,50,0.768306,0.560000,56.000000
24,Index,Conj Jupiter-Asc,37,0.140964,0.540541,54.054054
13,FX,Conj Jupiter-MC,26,0.154706,0.538462,53.846154


In [60]:
# Настройки для сырьевых товаров
COMMODITY_ASSETS = {'GLD': 'Gold', 'SLV': 'Silver', 'USO': 'Oil'}

def run_commodity_research(days=150):
    all_hits = []
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)

    for ticker, name in COMMODITY_ASSETS.items():
        print(f"🔍 Анализируем {name} ({ticker})...")
        data = yf.download(ticker, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='1h', progress=False, auto_adjust=True)
        if data.empty: continue

        for day, day_data in data.groupby(data.index.date):
            if len(day_data) < 5: continue
            hod_idx = day_data['High'].idxmax()
            lod_idx = day_data['Low'].idxmin()

            if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
            if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]

            for ext_label, ext_time in [('High (HOD)', hod_idx), ('Low (LOD)', lod_idx)]:
                obs = ephem.Observer()
                obs.lat, obs.lon = OBS_LAT, OBS_LON
                obs.date = ephem.Date(ext_time.to_pydatetime().astimezone(pytz.utc))
                mc, asc = get_angles_precise(obs)
                angles = {'Asc': asc, 'MC': mc}
                planets = {
                    "Sun": ephem.Ecliptic(ephem.Sun(obs)).lon / ephem.degree,
                    "Moon": ephem.Ecliptic(ephem.Moon(obs)).lon / ephem.degree,
                    "Jupiter": ephem.Ecliptic(ephem.Jupiter(obs)).lon / ephem.degree,
                    "Venus": ephem.Ecliptic(ephem.Venus(obs)).lon / ephem.degree # Добавим Венеру для золота/серебра
                }
                for p_name, p_lon in planets.items():
                    for a_name, a_lon in angles.items():
                        diff = abs(p_lon - a_lon) % 360
                        if diff > 180: diff = 360 - diff
                        for asp_val, asp_name in {0: 'Conj', 90: 'Square', 180: 'Opp'}.items():
                            if abs(diff - asp_val) <= TEST_ORB:
                                all_hits.append({'Asset': ticker, 'Extreme': ext_label, 'Aspect': f"{asp_name} {p_name}-{a_name}"})

    df_res = pd.DataFrame(all_hits)
    if df_res.empty: return None
    return df_res.groupby(['Asset', 'Extreme', 'Aspect']).size().reset_index(name='Frequency').sort_values(by=['Asset', 'Frequency'], ascending=[True, False])

commodity_results = run_commodity_research(150)
if commodity_results is not None:
    print("\n✅ ИССЛЕДОВАНИЕ СЫРЬЯ ЗАВЕРШЕНО")
    display(commodity_results.head(30))
else:
    print("Данные не найдены.")

🔍 Анализируем Gold (GLD)...


/tmp/ipykernel_5018/1349642872.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
/tmp/ipykernel_5018/1349642872.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]


🔍 Анализируем Silver (SLV)...


/tmp/ipykernel_5018/1349642872.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
/tmp/ipykernel_5018/1349642872.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]


🔍 Анализируем Oil (USO)...


/tmp/ipykernel_5018/1349642872.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
/tmp/ipykernel_5018/1349642872.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]



✅ ИССЛЕДОВАНИЕ СЫРЬЯ ЗАВЕРШЕНО


,Asset,Extreme,Aspect,Frequency
21,GLD,Low (LOD),Square Venus-Asc,8
19,GLD,Low (LOD),Square Moon-Asc,6
2,GLD,High (HOD),Conj Venus-MC,5
8,GLD,High (HOD),Square Moon-Asc,5
10,GLD,High (HOD),Square Venus-Asc,5
12,GLD,Low (LOD),Conj Moon-MC,5
13,GLD,Low (LOD),Conj Venus-MC,5
14,GLD,Low (LOD),Opp Jupiter-MC,5
17,GLD,Low (LOD),Square Jupiter-Asc,4
0,GLD,High (HOD),Conj Jupiter-Asc,3


### 📊 Исследование Криптовалют: Астро-сигналы vs Экстремумы (BTC, ETH, SOL, XRP, BNB, DOGE)
Анализ корреляции аспектов для основных цифровых активов за последние 150 дней.

In [61]:
# Список криптовалют для анализа
CRYPTO_ASSETS = {
    'BTC-USD': 'Bitcoin',
    'ETH-USD': 'Ethereum',
    'SOL-USD': 'Solana',
    'XRP-USD': 'XRP',
    'BNB-USD': 'BNB',
    'DOGE-USD': 'Dogecoin'
}

def run_crypto_research(days=150):
    all_hits = []
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)

    for ticker, name in CRYPTO_ASSETS.items():
        print(f"🔍 Анализируем {name} ({ticker})...")
        # Для крипто используем 1h данные. Yahoo предоставляет 1h данные максимум за 730 дней.
        data = yf.download(ticker, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='1h', progress=False, auto_adjust=True)
        if data.empty: continue

        for day, day_data in data.groupby(data.index.date):
            if len(day_data) < 12: continue # Фильтр на полные торговые сутки

            hod_idx = day_data['High'].idxmax()
            lod_idx = day_data['Low'].idxmin()

            if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
            if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]

            for ext_label, ext_time in [('High (HOD)', hod_idx), ('Low (LOD)', lod_idx)]:
                obs = ephem.Observer()
                obs.lat, obs.lon = OBS_LAT, OBS_LON
                obs.date = ephem.Date(ext_time.to_pydatetime().astimezone(pytz.utc))

                mc, asc = get_angles_precise(obs)
                angles = {'Asc': asc, 'MC': mc}

                planets = {
                    "Sun": ephem.Ecliptic(ephem.Sun(obs)).lon / ephem.degree,
                    "Moon": ephem.Ecliptic(ephem.Moon(obs)).lon / ephem.degree,
                    "Jupiter": ephem.Ecliptic(ephem.Jupiter(obs)).lon / ephem.degree
                }

                for p_name, p_lon in planets.items():
                    for a_name, a_lon in angles.items():
                        diff = abs(p_lon - a_lon) % 360
                        if diff > 180: diff = 360 - diff

                        for asp_val, asp_name in {0: 'Conj', 90: 'Square', 180: 'Opp'}.items():
                            if abs(diff - asp_val) <= TEST_ORB:
                                all_hits.append({
                                    'Asset': ticker,
                                    'Extreme': ext_label,
                                    'Aspect': f"{asp_name} {p_name}-{a_name}"
                                })

    df_res = pd.DataFrame(all_hits)
    if df_res.empty: return None
    return df_res.groupby(['Asset', 'Extreme', 'Aspect']).size().reset_index(name='Frequency').sort_values(by=['Asset', 'Frequency'], ascending=[True, False])

crypto_results = run_crypto_research(150)
if crypto_results is not None:
    print("\n✅ ИССЛЕДОВАНИЕ КРИПТО ЗАВЕРШЕНО")
    display(crypto_results.head(40))
else:
    print("Данные не найдены.")

🔍 Анализируем Bitcoin (BTC-USD)...


/tmp/ipykernel_5018/4250845734.py:28: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
/tmp/ipykernel_5018/4250845734.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]


🔍 Анализируем Ethereum (ETH-USD)...


/tmp/ipykernel_5018/4250845734.py:28: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
/tmp/ipykernel_5018/4250845734.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]


🔍 Анализируем Solana (SOL-USD)...


...

/tmp/ipykernel_5018/4250845734.py:28: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
/tmp/ipykernel_5018/4250845734.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]


🔍 Анализируем Dogecoin (DOGE-USD)...


/tmp/ipykernel_5018/4250845734.py:28: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(hod_idx, (pd.Series, pd.Index, list)): hod_idx = hod_idx[0]
/tmp/ipykernel_5018/4250845734.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(lod_idx, (pd.Series, pd.Index, list)): lod_idx = lod_idx[0]



✅ ИССЛЕДОВАНИЕ КРИПТО ЗАВЕРШЕНО


,Asset,Extreme,Aspect,Frequency
35,BNB-USD,Low (LOD),Square Sun-MC,13
17,BNB-USD,High (HOD),Square Sun-MC,12
10,BNB-USD,High (HOD),Opp Sun-Asc,11
28,BNB-USD,Low (LOD),Opp Sun-Asc,11
1,BNB-USD,High (HOD),Conj Jupiter-MC,9
34,BNB-USD,Low (LOD),Square Sun-Asc,9
12,BNB-USD,High (HOD),Square Jupiter-Asc,8
16,BNB-USD,High (HOD),Square Sun-Asc,8
13,BNB-USD,High (HOD),Square Jupiter-MC,7
14,BNB-USD,High (HOD),Square Moon-Asc,7


In [56]:
import yfinance as yf
import pandas as pd
import ephem
from datetime import datetime, timedelta
import pytz
import math

FX_PAIRS = ['EURUSD=X', 'GBPUSD=X', 'USDCHF=X', 'USDCAD=X', 'USDJPY=X', 'AUDUSD=X', 'NZDUSD=X']
NY_TZ = pytz.timezone('America/New_York')
OBS_LAT, OBS_LON = '40.7128', '-74.0060'
TEST_ORB = 4.0 # Умеренный допуск в 4 градуса

def get_angles_precise(obs):
    ra_mc = obs.sidereal_time()
    eps = 23.439 * math.pi / 180
    mc_lon = math.atan2(math.sin(ra_mc), math.cos(ra_mc) * math.cos(eps))
    lat = float(obs.lat) * math.pi / 180
    asc_lon = math.atan2(math.cos(ra_mc), -(math.sin(ra_mc) * math.cos(eps) + math.tan(lat) * math.sin(eps)))
    return (mc_lon * 180 / math.pi) % 360, (asc_lon * 180 / math.pi) % 360

def analyze_fx_extremes(days=120):
    results = []
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)

    for ticker in FX_PAIRS:
        print(f"Обработка {ticker}...")
        data = yf.download(ticker, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='1h', progress=False, auto_adjust=True)
        if data.empty: continue

        # Группируем по дням для поиска HOD/LOD
        days_group = data.groupby(data.index.date)
        for day, day_data in days_group:
            if len(day_data) < 12: continue

            # Исправляем извлечение времени (берем первое значение, если вернулась серия)
            hod_time = day_data['High'].idxmax()
            if isinstance(hod_time, pd.Series): hod_time = hod_time.iloc[0]

            lod_time = day_data['Low'].idxmin()
            if isinstance(lod_time, pd.Series): lod_time = lod_time.iloc[0]

            for ext_type, ext_time in [('High of Day', hod_time), ('Low of Day', lod_time)]:
                obs = ephem.Observer()
                obs.lat, obs.lon = OBS_LAT, OBS_LON
                obs.date = ephem.Date(ext_time.to_pydatetime().astimezone(pytz.utc))

                mc, asc = get_angles_precise(obs)
                planets = {"Sun": ephem.Sun(obs), "Moon": ephem.Moon(obs), "Jupiter": ephem.Jupiter(obs)}

                for p_name, p_obj in planets.items():
                    p_lon = ephem.Ecliptic(p_obj).lon / ephem.degree
                    for a_name, a_lon in [('Asc', asc), ('MC', mc)]:
                        diff = abs(p_lon - a_lon) % 360
                        if diff > 180: diff = 360 - diff

                        for asp_val, asp_name in {0: 'Conj', 90: 'Square', 180: 'Opp'}.items():
                            if abs(diff - asp_val) <= TEST_ORB:
                                results.append({
                                    'Pair': ticker,
                                    'Extreme_Type': ext_type,
                                    'Aspect': f"{asp_name} {p_name}-{a_name}",
                                    'Count': 1
                                })

    return pd.DataFrame(results)

# Запуск анализа
res_fx_ext = analyze_fx_extremes(120)

if not res_fx_ext.empty:
    summary = res_fx_ext.groupby(['Pair', 'Extreme_Type', 'Aspect']).size().reset_index(name='Frequency')
    print("\n--- ТОП-20 Наиболее устойчивых связей (Аспект -> Дневной Экстремум) ---")
    display(summary.sort_values(by='Frequency', ascending=False).head(20))
else:
    print("Сигналы не найдены.")

Обработка EURUSD=X...
Обработка GBPUSD=X...
Обработка USDCHF=X...
Обработка USDCAD=X...
Обработка USDJPY=X...
Обработка AUDUSD=X...
Обработка NZDUSD=X...

--- ТОП-20 Наиболее устойчивых связей (Аспект -> Дневной Экстремум) ---


,Pair,Extreme_Type,Aspect,Frequency
100,GBPUSD=X,Low of Day,Square Sun-MC,11
50,EURUSD=X,High of Day,Square Sun-MC,10
219,USDJPY=X,High of Day,Square Sun-MC,10
31,AUDUSD=X,Low of Day,Square Sun-Asc,9
118,NZDUSD=X,High of Day,Square Sun-MC,9
117,NZDUSD=X,High of Day,Square Sun-Asc,8
20,AUDUSD=X,Low of Day,Conj Sun-MC,8
186,USDCHF=X,High of Day,Square Sun-MC,8
114,NZDUSD=X,High of Day,Square Jupiter-MC,8
13,AUDUSD=X,High of Day,Square Sun-Asc,8


In [50]:
import yfinance as yf
import pandas as pd
import ephem
from datetime import datetime, timedelta
import pytz
import math

# Настройки
ASSETS = ["SPY", "BTC-USD", "EURUSD=X", "GBPUSD=X"]
TEST_ORB = 5
NY_TZ = pytz.timezone('America/New_York')
OBSERVER_LAT = '40.7128'
OBSERVER_LON = '-74.0060'

def get_angles_ecliptic(obs):
    ra_mc = obs.sidereal_time()
    eps = 23.439 * math.pi / 180
    mc_lon = math.atan2(math.sin(ra_mc), math.cos(ra_mc) * math.cos(eps))
    lat = float(obs.lat) * math.pi / 180
    asc_lon = math.atan2(math.cos(ra_mc), -(math.sin(ra_mc) * math.cos(eps) + math.tan(lat) * math.sin(eps)))
    return (mc_lon * 180 / math.pi) % 360, (asc_lon * 180 / math.pi) % 360

def run_period_backtest(period_days):
    end_date = datetime.now(NY_TZ)
    start_date = end_date - timedelta(days=period_days)

    all_results = []

    for ticker in ASSETS:
        df = yf.download(ticker, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='1d', progress=False, auto_adjust=True)
        if df.empty: continue

        # Handle MultiIndex if necessary
        if isinstance(df.columns, pd.MultiIndex):
            prices = df['Close'][ticker]
        else:
            prices = df['Close']

        obs = ephem.Observer()
        obs.lat, obs.lon = OBSERVER_LAT, OBSERVER_LON

        for day in prices.index:
            # Check at NYSE open 9:30 AM NY Time
            check_time = NY_TZ.localize(day.replace(hour=9, minute=30)).astimezone(pytz.utc)
            obs.date = ephem.Date(check_time)
            mc, asc = get_angles_ecliptic(obs)

            planets = {
                "Sun": ephem.Ecliptic(ephem.Sun(obs)).lon / ephem.degree,
                "Moon": ephem.Ecliptic(ephem.Moon(obs)).lon / ephem.degree,
                "Jupiter": ephem.Ecliptic(ephem.Jupiter(obs)).lon / ephem.degree
            }

            for p_name, p_lon in planets.items():
                for a_name, a_lon in [("Asc", asc), ("MC", mc)]:
                    diff = abs(p_lon - a_lon) % 360
                    if diff > 180: diff = 360 - diff

                    for asp_val, asp_name in {0: "Conj", 90: "Square", 180: "Opp"}.items():
                        if abs(diff - asp_val) <= TEST_ORB:
                            try:
                                idx = prices.index.get_loc(day)
                                if idx + 1 < len(prices):
                                    p_now = float(prices.iloc[idx])
                                    p_next = float(prices.iloc[idx+1])
                                    ret = ((p_next - p_now) / p_now) * 100
                                    all_results.append({
                                        'Asset': ticker,
                                        'Aspect': f"{asp_name} {p_name}-{a_name}",
                                        'Return_%': ret
                                    })
                            except: continue

    res_df = pd.DataFrame(all_results)
    if res_df.empty: return None

    summary = res_df.groupby(['Asset', 'Aspect'])['Return_%'].agg(['count', 'mean'])
    summary['Win_Rate'] = res_df[res_df['Return_%'] > 0].groupby(['Asset', 'Aspect'])['Return_%'].count() / summary['count']
    return summary.sort_values(by='mean', ascending=False)

print("--- БЭКТЕСТ ЗА ГОД (365 дней) ---")
display(run_period_backtest(365))

print("\n--- БЭКТЕСТ ЗА КВАРТАЛ (90 дней) ---")
display(run_period_backtest(90))

print("\n--- БЭКТЕСТ ЗА МЕСЯЦ (30 дней) ---")
display(run_period_backtest(30))

--- БЭКТЕСТ ЗА ГОД (365 дней) ---


count      mean  Win_Rate
Asset    Aspect                                       
BTC-USD  Conj Jupiter-Asc        5  1.497311  0.600000
SPY      Conj Jupiter-Asc        4  1.073668  0.750000
BTC-USD  Opp Moon-MC            10  0.721070  0.600000
SPY      Square Jupiter-MC      10  0.572559  0.700000
         Opp Moon-MC             6  0.515584  1.000000
         Opp Moon-Asc            6  0.515577  0.666667
         Square Moon-Asc        14  0.384232  0.714286
         Opp Jupiter-Asc         7  0.380569  0.714286
BTC-USD  Opp Moon-Asc           10  0.375994  0.600000
EURUSD=X Conj Moon-Asc           6  0.322684  0.666667
SPY      Conj Moon-MC            8  0.255354  0.500000
GBPUSD=X Conj Moon-Asc           6  0.222699  0.666667
         Opp Moon-Asc            6  0.201931  0.833333
         Conj Moon-MC            8  0.198468  0.500000
EURUSD=X Opp Moon-Asc            6  0.190071  0.833333
BTC-USD  Square Moon-Asc        22  0.186682  0.500000
         Conj Jupiter-MC        14  0.134045  0.571429
EURUSD=X Conj Moon-MC            8  0.111054  0.375000
SPY      Conj Jupiter-MC         9  0.110781  0.666667
         Conj Moon-Asc           6  0.088648  0.500000
GBPUSD=X Opp Jupiter-Asc         8  0.088070  0.625000
         Square Moon-Asc        14  0.073723  0.500000
EURUSD=X Square Moon-MC          9  0.052069  0.555556
         Opp Jupiter-Asc         8  0.022158  0.500000
         Conj Jupiter-MC        10  0.003808  0.700000
BTC-USD  Conj Moon-Asc          10  0.003581  0.400000
SPY      Square Moon-MC          9  0.003556  0.666667
GBPUSD=X Square Moon-MC          9  0.002486  0.555556
EURUSD=X Square Moon-Asc        14  0.000656  0.428571
GBPUSD=X Conj Jupiter-Asc        4 -0.005643  0.500000
         Conj Jupiter-MC        10 -0.012418  0.700000
EURUSD=X Square Jupiter-Asc     12 -0.022491  0.500000
         Opp Jupiter-MC          8 -0.038627  0.250000
BTC-USD  Opp Jupiter-Asc        10 -0.039109  0.400000
GBPUSD=X Square Jupiter-MC      10 -0.043126  0.500000
BTC-USD  Square Moon-MC         17 -0.046519  0.470588
GBPUSD=X Square Jupiter-Asc     12 -0.054410  0.500000
         Opp Moon-MC             6 -0.056194  0.500000
EURUSD=X Opp Moon-MC             6 -0.081681  0.333333
BTC-USD  Conj Moon-MC           11 -0.103709  0.454545
GBPUSD=X Opp Jupiter-MC          8 -0.103747  0.250000
SPY      Square Jupiter-Asc     11 -0.126787  0.545455
BTC-USD  Square Jupiter-MC      16 -0.154423  0.375000
EURUSD=X Square Jupiter-MC      10 -0.156167  0.300000
         Conj Jupiter-Asc        4 -0.184820  0.500000
BTC-USD  Square Jupiter-Asc     19 -0.189214  0.473684
SPY      Opp Jupiter-MC          7 -0.194269  0.428571
BTC-USD  Opp Jupiter-MC         11 -0.420918  0.363636


--- БЭКТЕСТ ЗА КВАРТАЛ (90 дней) ---


count      mean  Win_Rate
Asset    Aspect                                       
SPY      Conj Moon-MC            1  1.218488  1.000000
GBPUSD=X Conj Moon-MC            1  0.902826  1.000000
EURUSD=X Conj Moon-MC            1  0.802638  1.000000
GBPUSD=X Opp Moon-Asc            1  0.651864  1.000000
         Square Moon-Asc         2  0.536905  1.000000
SPY      Opp Moon-Asc            1  0.482183  1.000000
EURUSD=X Square Moon-Asc         2  0.474007  1.000000
         Opp Moon-Asc            1  0.406802  1.000000
         Conj Moon-Asc           1  0.367673  1.000000
SPY      Square Moon-Asc         2  0.365947  0.500000
GBPUSD=X Conj Moon-Asc           1  0.347752  1.000000
BTC-USD  Conj Moon-MC            3  0.133779  0.333333
EURUSD=X Square Jupiter-Asc      5  0.070839  0.200000
BTC-USD  Conj Moon-Asc           2  0.063282  0.500000
GBPUSD=X Square Jupiter-Asc      5  0.057441  0.400000
EURUSD=X Opp Jupiter-MC          8 -0.038627  0.250000
BTC-USD  Square Jupiter-Asc      8 -0.068700  0.375000
GBPUSD=X Opp Jupiter-MC          8 -0.103747  0.250000
SPY      Opp Jupiter-MC          7 -0.194269  0.428571
         Square Jupiter-Asc      5 -0.319983  0.400000
BTC-USD  Opp Jupiter-MC         11 -0.420918  0.363636
         Square Moon-Asc         6 -0.556537  0.166667
SPY      Conj Moon-Asc           1 -0.654695       NaN
BTC-USD  Square Moon-MC          4 -0.970195       NaN
         Opp Moon-Asc            2 -0.990907       NaN
         Opp Moon-MC             2 -1.213695       NaN


--- БЭКТЕСТ ЗА МЕСЯЦ (30 дней) ---


count      mean  Win_Rate
Asset    Aspect                                    
SPY      Conj Moon-MC         1  1.218488       1.0
GBPUSD=X Conj Moon-MC         1  0.902826       1.0
EURUSD=X Conj Moon-MC         1  0.802638       1.0
BTC-USD  Conj Moon-Asc        1  0.632970       1.0
GBPUSD=X Square Moon-Asc      2  0.536905       1.0
EURUSD=X Square Moon-Asc      2  0.474007       1.0
         Conj Moon-Asc        1  0.367673       1.0
SPY      Square Moon-Asc      2  0.365947       0.5
GBPUSD=X Conj Moon-Asc        1  0.347752       1.0
BTC-USD  Conj Moon-MC         1 -0.406837       NaN
SPY      Conj Moon-Asc        1 -0.654695       NaN
BTC-USD  Square Moon-Asc      2 -0.860002       NaN

In [51]:
print("--- БЭКТЕСТ ЗА ПОСЛЕДНИЕ 7 ДНЕЙ ---")
display(run_period_backtest(7))

--- БЭКТЕСТ ЗА ПОСЛЕДНИЕ 7 ДНЕЙ ---


None

In [52]:
# Настройка только для Форекс
ASSETS_FOREX = ['EURUSD=X', 'GBPUSD=X']

def run_forex_only_backtest(period_days):
    # Временно подменяем глобальные активы
    global ASSETS
    original_assets = ASSETS
    ASSETS = ASSETS_FOREX

    result = run_period_backtest(period_days)

    # Возвращаем настройки
    ASSETS = original_assets
    return result

print('--- БЭКТЕСТ FOREX (EUR/USD, GBP/USD) ЗА 30 ДНЕЙ ---')
display(run_forex_only_backtest(30))

--- БЭКТЕСТ FOREX (EUR/USD, GBP/USD) ЗА 30 ДНЕЙ ---


count      mean  Win_Rate
Asset    Aspect                                    
GBPUSD=X Conj Moon-MC         1  0.902826       1.0
EURUSD=X Conj Moon-MC         1  0.802638       1.0
GBPUSD=X Square Moon-Asc      2  0.536905       1.0
EURUSD=X Square Moon-Asc      2  0.474007       1.0
         Conj Moon-Asc        1  0.367673       1.0
GBPUSD=X Conj Moon-Asc        1  0.347752       1.0

### Ежедневный сканер разворотных точек
Этот скрипт рассчитывает точное время (NY Time), когда планеты образуют аспекты к углам карты (Asc/MC) в течение ближайших суток.

In [40]:
import ephem
from datetime import datetime, timedelta
import pytz
import pandas as pd

# Настройки
NY_TZ = pytz.timezone('America/New_York')
OBSERVER_LAT = '40.7128'
OBSERVER_LON = '-74.0060'
ORB = 1.0  # Ищем точные моменты (допуск 1 градус)

def get_angles(obs):
    # Упрощенный расчет эклиптических углов для поиска транзитов
    ra_mc = obs.sidereal_time()
    mc_lon = (ra_mc * 15 / ephem.degree) % 360 # Приблизительно
    # Асцендент требует более сложной формулы, используем radec_of для точки горизонта
    asc_lon = (obs.radec_of(0, 0)[0] / ephem.degree) % 360
    return mc_lon, asc_lon

def scan_day_for_reversals():
    obs = ephem.Observer()
    obs.lat, obs.lon = OBSERVER_LAT, OBSERVER_LON

    start_scan = datetime.now(NY_TZ)
    print(f"--- Прогноз разворотных точек на: {start_scan.strftime('%Y-%m-%d')} ---\n")

    found_events = []

    # Сканируем каждые 5 минут в течение 24 часов
    for i in range(0, 1440, 5):
        current_time = start_scan + timedelta(minutes=i)
        obs.date = ephem.Date(current_time.astimezone(pytz.utc))

        mc, asc = get_angles(obs)
        planets = {
            "Sun": ephem.Ecliptic(ephem.Sun(obs)).lon / ephem.degree,
            "Moon": ephem.Ecliptic(ephem.Moon(obs)).lon / ephem.degree,
            "Jupiter": ephem.Ecliptic(ephem.Jupiter(obs)).lon / ephem.degree
        }

        for p_name, p_lon in planets.items():
            for angle_name, angle_lon in [("Asc", asc), ("MC", mc)]:
                diff = abs(p_lon - angle_lon) % 360
                if diff > 180: diff = 360 - diff

                for asp_val, asp_name in {0: "Conj", 90: "Square", 180: "Opp"}.items():
                    if abs(diff - asp_val) < 0.5: # Точное касание
                        event_key = f"{asp_name}_{p_name}_{angle_name}"
                        # Записываем только первый момент касания
                        if not found_events or found_events[-1]['Key'] != event_key:
                            found_events.append({
                                'Time': current_time.strftime('%H:%M'),
                                'Event': f"{asp_name} {p_name} to {angle_name}",
                                'Key': event_key
                            })

    if found_events:
        df = pd.DataFrame(found_events)[['Time', 'Event']]
        display(df)
    else:
        print("На ближайшие сутки точных аспектных касаний не найдено.")

scan_day_for_reversals()

--- Прогноз разворотных точек на: 2026-05-06 ---



,Time,Event
0,05:16,Square Jupiter to MC
1,07:01,Square Sun to Asc
2,10:36,Square Moon to Asc
3,10:51,Square Jupiter to MC
4,11:16,Square Jupiter to Asc
5,11:46,Conj Sun to MC
6,13:01,Opp Sun to Asc
7,16:46,Conj Moon to Asc
8,17:16,Opp Jupiter to Asc
9,17:21,Opp Sun to MC


### Глобальный бэктест интрадей разворотов (730 дней)
Анализ корреляции аспектов к углам с разворотами на акциях, золоте и крипте.

In [43]:
import yfinance as yf
import pandas as pd
import ephem
from datetime import datetime, timedelta
import pytz
import math

# Параметры
ASSETS = ["SPY", "QQQ", "GLD", "BTC-USD"]
TEST_ORB = 8
NY_TZ = pytz.timezone('America/New_York')
OBSERVER_LAT = '40.7128'
OBSERVER_LON = '-74.0060'

def get_angles_ecliptic(obs):
    ra_mc = obs.sidereal_time()
    eps = 23.439 * math.pi / 180
    mc_lon = math.atan2(math.sin(ra_mc), math.cos(ra_mc) * math.cos(eps))
    lat = float(obs.lat) * math.pi / 180
    asc_lon = math.atan2(math.cos(ra_mc), -(math.sin(ra_mc) * math.cos(eps) + math.tan(lat) * math.sin(eps)))
    return (mc_lon * 180 / math.pi) % 360, (asc_lon * 180 / math.pi) % 360

def run_global_backtest():
    # Используем фиксированный период 2023-2024 на дневных данных
    start_date = "2023-01-01"
    end_date = "2024-06-01"

    all_results = []

    for ticker in ASSETS:
        print(f"Обработка {ticker}...")
        # Используем '1d' для обхода лимитов Yahoo на старые данные
        df = yf.download(ticker, start=start_date, end=end_date, interval='1d', progress=False, auto_adjust=True)
        if df.empty: continue

        obs = ephem.Observer()
        obs.lat, obs.lon = OBSERVER_LAT, OBSERVER_LON

        for current_day in df.index:
            # Исправление: локализуем naive timestamp перед конвертацией в UTC
            naive_time = current_day.replace(hour=9, minute=30)
            check_time = NY_TZ.localize(naive_time).astimezone(pytz.utc)
            obs.date = ephem.Date(check_time)

            mc, asc = get_angles_ecliptic(obs)
            planets = {
                "Sun": ephem.Ecliptic(ephem.Sun(obs)).lon / ephem.degree,
                "Moon": ephem.Ecliptic(ephem.Moon(obs)).lon / ephem.degree,
                "Jupiter": ephem.Ecliptic(ephem.Jupiter(obs)).lon / ephem.degree
            }

            for p_name, p_lon in planets.items():
                for a_name, a_lon in [("Asc", asc), ("MC", mc)]:
                    diff = abs(p_lon - a_lon) % 360
                    if diff > 180: diff = 360 - diff

                    for asp_val, asp_name in {0: "Conj", 90: "Square", 180: "Opp"}.items():
                        if abs(diff - asp_val) <= TEST_ORB:
                            try:
                                idx = df.index.get_loc(current_day)
                                if idx + 1 < len(df):
                                    p_now = df.iloc[idx]['Close']
                                    p_next = df.iloc[idx+1]['Close']
                                    all_results.append({
                                        'Asset': ticker,
                                        'Aspect': f"{asp_name} {p_name}-{a_name}",
                                        'Return_%': ((p_next - p_now) / p_now) * 100
                                    })
                            except: continue

    final_df = pd.DataFrame(all_results)
    if not final_df.empty:
        report = final_df.groupby(['Asset', 'Aspect'])['Return_%'].agg(['count', 'mean']).rename(columns={'mean': 'Avg_NextDay_Ret_%'})
        print("\n--- Топ-20 наиболее прибыльных аспектных паттернов ---")
        display(report.sort_values(by='Avg_NextDay_Ret_%', ascending=False).head(20))
    else:
        print("Сигналы не найдены.")

run_global_backtest()

Обработка SPY...
Обработка QQQ...
Обработка GLD...
Обработка BTC-USD...

--- Топ-20 наиболее прибыльных аспектных паттернов ---


/tmp/ipykernel_5018/3108366195.py:73: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  report = final_df.groupby(['Asset', 'Aspect'])['Return_%'].agg(['count', 'mean']).rename(columns={'mean': 'Avg_NextDay_Ret_%'})


count Avg_NextDay_Ret_%
Asset   Aspect                                     
BTC-USD Opp Jupiter-MC         15           1.00333
        Opp Moon-Asc           22          0.974086
        Square Jupiter-Asc     35           0.92308
        Conj Jupiter-MC        19           0.79831
        Square Moon-MC         50          0.768306
        Conj Jupiter-Asc       50          0.385548
QQQ     Conj Moon-MC           16          0.372427
BTC-USD Opp Moon-MC            29          0.298967
GLD     Conj Jupiter-Asc       37          0.292925
QQQ     Conj Jupiter-MC        12          0.265908
        Square Moon-Asc        33          0.255793
SPY     Conj Moon-MC           16          0.250276
        Square Jupiter-Asc     24          0.241489
QQQ     Square Jupiter-Asc     24          0.232497
BTC-USD Opp Jupiter-Asc        15          0.177079
QQQ     Conj Jupiter-Asc       37           0.17333
SPY     Conj Jupiter-MC        12          0.168806
        Opp Jupiter-MC         10           0.16483
QQQ     Opp Jupiter-MC         10          0.164024
SPY     Square Moon-Asc        33           0.15567

In [48]:
import yfinance as yf
import pandas as pd
import ephem
from datetime import datetime, timedelta
import pytz
import math

# Настройки
ASSETS = ["EURUSD=X", "GBPUSD=X", "SPY", "BTC-USD"]
TEST_ORB = 8
NY_TZ = pytz.timezone('America/New_York')
OBSERVER_LAT = '40.7128'
OBSERVER_LON = '-74.0060'

def get_angles_ecliptic(obs):
    ra_mc = obs.sidereal_time()
    eps = 23.439 * math.pi / 180
    mc_lon = math.atan2(math.sin(ra_mc), math.cos(ra_mc) * math.cos(eps))
    lat = float(obs.lat) * math.pi / 180
    asc_lon = math.atan2(math.cos(ra_mc), -(math.sin(ra_mc) * math.cos(eps) + math.tan(lat) * math.sin(eps)))
    return (mc_lon * 180 / math.pi) % 360, (asc_lon * 180 / math.pi) % 360

all_results = []
for ticker in ASSETS:
    print(f"Обработка {ticker}...")
    df = yf.download(ticker, start="2023-01-01", end="2024-06-01", interval='1d', progress=False, auto_adjust=True)
    if df.empty: continue

    # Если yfinance вернул MultiIndex (когда несколько тикеров или специфичный формат), берем Close
    if isinstance(df.columns, pd.MultiIndex):
        prices = df['Close'][ticker]
    else:
        prices = df['Close']

    obs = ephem.Observer()
    obs.lat, obs.lon = OBSERVER_LAT, OBSERVER_LON

    for current_day in prices.index:
        naive_time = current_day.replace(hour=9, minute=30)
        check_time = NY_TZ.localize(naive_time).astimezone(pytz.utc)
        obs.date = ephem.Date(check_time)
        mc, asc = get_angles_ecliptic(obs)

        planets = {
            "Sun": ephem.Ecliptic(ephem.Sun(obs)).lon / ephem.degree,
            "Moon": ephem.Ecliptic(ephem.Moon(obs)).lon / ephem.degree,
            "Jupiter": ephem.Ecliptic(ephem.Jupiter(obs)).lon / ephem.degree
        }

        for p_name, p_lon in planets.items():
            for a_name, a_lon in [("Asc", asc), ("MC", mc)]:
                diff = abs(p_lon - a_lon) % 360
                if diff > 180: diff = 360 - diff

                for asp_val, asp_name in {0: "Conj", 90: "Square", 180: "Opp"}.items():
                    if abs(diff - asp_val) <= TEST_ORB:
                        try:
                            idx = prices.index.get_loc(current_day)
                            if idx + 1 < len(prices):
                                p_now = float(prices.iloc[idx])
                                p_next = float(prices.iloc[idx+1])
                                ret = ((p_next - p_now) / p_now) * 100
                                all_results.append({
                                    'Asset': ticker,
                                    'Aspect': f'{asp_name} {p_name}-{a_name}',
                                    'Return_%': ret,
                                    'Is_Win': 1 if ret > 0 else 0
                                })
                        except:
                            continue

final_df = pd.DataFrame(all_results)
if not final_df.empty:
    report = final_df.groupby(['Asset', 'Aspect']).agg(
        Count=('Return_%', 'count'),
        Avg_Ret=('Return_%', 'mean'),
        Win_Rate=('Is_Win', 'mean')
    )
    report['Win_Rate_%'] = report['Win_Rate'] * 100

    print("\n--- ИТОГОВЫЙ ОТЧЕТ (Win Rate & Return) ---")
    display(report[report['Count'] > 10].sort_values(by='Win_Rate_%', ascending=False).head(30))
else:
    print("Сигналы не найдены.")

Обработка EURUSD=X...
Обработка GBPUSD=X...
Обработка SPY...
Обработка BTC-USD...

--- ИТОГОВЫЙ ОТЧЕТ (Win Rate & Return) ---


Count   Avg_Ret  Win_Rate  Win_Rate_%
Asset    Aspect                                                   
GBPUSD=X Conj Moon-MC           16  0.416976  0.937500   93.750000
EURUSD=X Conj Moon-MC           16  0.240165  0.875000   87.500000
SPY      Square Jupiter-Asc     24  0.241489  0.666667   66.666667
GBPUSD=X Square Moon-Asc        33  0.121139  0.666667   66.666667
BTC-USD  Square Jupiter-Asc     35  0.923080  0.657143   65.714286
SPY      Square Moon-Asc        33  0.155670  0.636364   63.636364
         Conj Moon-MC           16  0.250276  0.625000   62.500000
EURUSD=X Square Moon-Asc        33  0.053630  0.606061   60.606061
SPY      Opp Moon-MC            20  0.077009  0.600000   60.000000
BTC-USD  Opp Jupiter-MC         15  1.003330  0.600000   60.000000
SPY      Conj Jupiter-MC        12  0.168806  0.583333   58.333333
BTC-USD  Square Moon-MC         50  0.768306  0.560000   56.000000
SPY      Conj Jupiter-Asc       37  0.140964  0.540541   54.054054
GBPUSD=X Conj Jupiter-MC        13  0.163920  0.538462   53.846154
EURUSD=X Conj Jupiter-MC        13  0.145492  0.538462   53.846154
SPY      Conj Moon-Asc          15 -0.038625  0.533333   53.333333
         Square Moon-MC         34 -0.004921  0.529412   52.941176
BTC-USD  Conj Jupiter-MC        19  0.798310  0.526316   52.631579
         Conj Jupiter-Asc       50  0.385548  0.520000   52.000000
         Square Jupiter-MC      64  0.068502  0.515625   51.562500
GBPUSD=X Square Jupiter-MC      47 -0.046092  0.510638   51.063830
BTC-USD  Opp Moon-Asc           22  0.974086  0.500000   50.000000
SPY      Square Jupiter-MC      46  0.007872  0.500000   50.000000
BTC-USD  Square Moon-Asc        46  0.100487  0.478261   47.826087
         Conj Moon-Asc          21  0.068240  0.476190   47.619048
GBPUSD=X Conj Jupiter-Asc       38 -0.056172  0.473684   47.368421
         Opp Moon-Asc           15 -0.036821  0.466667   46.666667
BTC-USD  Opp Jupiter-Asc        15  0.177079  0.466667   46.666667
GBPUSD=X Opp Moon-MC            20 -0.107192  0.450000   45.000000
BTC-USD  Opp Moon-MC            29  0.298967  0.448276   44.827586

In [13]:
!pip install -q yfinance

In [39]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import ephem
import pytz
import math

TEST_ORB = 8
ASPECT_TYPES = {0: "Conj", 90: "Square", 180: "Opp"}
NY_TZ = pytz.timezone('America/New_York')
OBSERVER_LAT = '40.7128'
OBSERVER_LON = '-74.0060'

def get_market_data(ticker, start_s, end_s):
    # Используем '1d' интервал, так как '1h' ограничен 730 днями
    df = yf.download(ticker, start=start_s, end=end_s, interval='1d', progress=False, auto_adjust=True)
    return df

def get_angles_ecliptic(obs):
    ra_mc = obs.sidereal_time()
    eps = 23.439 * math.pi / 180
    mc_lon = math.atan2(math.sin(ra_mc), math.cos(ra_mc) * math.cos(eps))
    lat = float(obs.lat) * math.pi / 180
    asc_lon = math.atan2(math.cos(ra_mc), -(math.sin(ra_mc) * math.cos(eps) + math.tan(lat) * math.sin(eps)))
    return (mc_lon * 180 / math.pi) % 360, (asc_lon * 180 / math.pi) % 360

def run_study():
    # Анализируем 2023-2024 годы на дневных данных
    start_dt = datetime(2023, 5, 1)
    end_dt = datetime(2024, 5, 1)
    market_data = get_market_data("SPY", "2023-01-01", "2024-06-01")

    if market_data.empty:
        print("Ошибка загрузки данных.")
        return

    obs = ephem.Observer()
    obs.lat, obs.lon = OBSERVER_LAT, OBSERVER_LON
    results = []

    curr = start_dt
    while curr < end_dt:
        if curr.weekday() < 5:
            # На дневных данных проверяем аспект на момент открытия (9:30)
            check_time = curr.replace(hour=9, minute=30)
            obs.date = ephem.Date(check_time)
            mc, asc = get_angles_ecliptic(obs)

            planets = {
                "Sun": ephem.Ecliptic(ephem.Sun(obs)).lon / ephem.degree,
                "Moon": ephem.Ecliptic(ephem.Moon(obs)).lon / ephem.degree,
                "Jupiter": ephem.Ecliptic(ephem.Jupiter(obs)).lon / ephem.degree,
                "Asc": asc, "MC": mc
            }

            for p in ["Sun", "Moon", "Jupiter"]:
                for angle_pt in ["Asc", "MC"]:
                    diff = abs(planets[p] - planets[angle_pt]) % 360
                    if diff > 180: diff = 360 - diff

                    for asp_val, asp_name in ASPECT_TYPES.items():
                        if abs(diff - asp_val) <= TEST_ORB:
                            try:
                                # Смотрим изменение цены Close относительно следующего торгового дня
                                dt_str = curr.strftime('%Y-%m-%d')
                                if dt_str in market_data.index:
                                    idx = market_data.index.get_loc(dt_str)
                                    p_now = market_data.iloc[idx]['Close']
                                    p_next = market_data.iloc[min(idx+1, len(market_data)-1)]['Close']
                                    results.append({
                                        'Aspect': f"{asp_name} {p}-{angle_pt}",
                                        'Chg_NextDay_%': round(((p_next - p_now) / p_now) * 100, 3)
                                    })
                            except: continue
        curr += timedelta(days=1)

    df_res = pd.DataFrame(results)
    if not df_res.empty:
        print(f"Анализ завершен. Найдено сигналов: {len(df_res)}")
        summary = df_res.groupby('Aspect')['Chg_NextDay_%'].agg(['count', 'mean']).rename(columns={'mean': 'Avg_Return_%'})
        display(summary.sort_values('count', ascending=False))
    else:
        print("Сигналы не обнаружены.")

run_study()

Анализ завершен. Найдено сигналов: 193


/tmp/ipykernel_5018/3400835063.py:80: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  summary = df_res.groupby('Aspect')['Chg_NextDay_%'].agg(['count', 'mean']).rename(columns={'mean': 'Avg_Return_%'})


,count,Avg_Return_%
Aspect,,
Square Jupiter-MC,26,0.195038
Square Moon-MC,25,0.19436
Square Jupiter-Asc,24,0.09
Square Moon-Asc,22,0.212273
Conj Jupiter-Asc,15,0.055533
Opp Moon-Asc,13,0.326385
Conj Moon-Asc,13,0.164462
Conj Moon-MC,13,0.079846
Opp Jupiter-MC,12,0.120917
